In [ ]:
from pathlib import Path
import filecmp
import os
import re

In [ ]:
OLD_DIR = "/data/jim/CMB_Data/Datasets/I_512_1750_d1_0rg/PyILC_CNILC_new_install/Pure_PyILC2"
NEW_DIR = "outputs"

In [ ]:
patterns = {
    "needletcoeff": re.compile(r"CN__needletcoeffmap_freq\d+_scale\d+\.fits"),
    "covmap": re.compile(r"CN__needletcoeff_covmap_freq\d+_freq\d+_scale\d+\.fits"),
    "per_scale_ilc": re.compile(r"CN_needletILCmap_scale\d+_component_.*_includechannels.*\.fits"),
    "final_ilc": re.compile(r"CN_needletILCmap_component_.*\.fits"),
}

def categorize(fname: str) -> str:
    for key, pat in patterns.items():
        if pat.fullmatch(fname):
            return key
    return "other"

for label, directory in [("OLD", OLD_DIR), ("NEW", NEW_DIR)]:
    files = [f for f in os.listdir(directory) if f.endswith(".fits")]
    counts = {k: 0 for k in list(patterns.keys()) + ["other"]}
    for f in files:
        cat = categorize(f)
        counts[cat] += 1
    print(f"\n{label} DIR: {directory}")
    for k, v in counts.items():
        print(f"  {k}: {v}")

In [ ]:
import healpy as hp
from cmbml.core.asset_handlers import HealpyMap
import matplotlib.pyplot as plt

In [ ]:
from utils.show_maps import show_obs_logNxN

In [ ]:
# final_pat = patterns["final_ilc"]

# old_final = [f for f in os.listdir(OLD_DIR) if final_pat.fullmatch(f)]
# new_final = [f for f in os.listdir(NEW_DIR) if final_pat.fullmatch(f)]

# final_path_orig = Path(OLD_DIR) / old_final[0] if old_final else None
# final_path_new  = Path(NEW_DIR) / new_final[0] if new_final else None

# final_pred_orig = HealpyMap().read(final_path_orig)[0]
# final_pred_new  = HealpyMap().read(final_path_new)[0] * 1e6

# show_obs_logNxN([final_pred_orig, final_pred_new, final_pred_new - final_pred_orig], 
#                 labels=["Old", "New", "Delta"],
#                 min_val=-400, max_val=400,
#                 n_cols=3, n_rows=1, 
#                 xsize=1200, dpi=100, 
#                 )

In [ ]:
def make_fn(kind: str, **kwargs) -> str:
    """
    Generate PyILC-style filenames.

    kind can be one of:
      - "needletcoeff"
      - "covmap"
      - "per_scale_ilc"
      - "final_ilc"

    kwargs depend on kind:
      needletcoeff: f_orig, s
      covmap: i_orig, j_orig, s
      per_scale_ilc: s, comp, includechannels (string of indices)
      final_ilc: comp
    """
    if kind == "needletcoeff":
        return f"CN__needletcoeffmap_freq{kwargs['f_orig']}_scale{kwargs['s']}.fits"

    elif kind == "covmap":
        return f"CN__needletcoeff_covmap_freq{kwargs['i_orig']}_freq{kwargs['j_orig']}_scale{kwargs['s']}.fits"

    elif kind == "per_scale_ilc":
        return (
            f"CN_needletILCmap_scale{kwargs['s']}_component_{kwargs['comp']}"
            f"_includechannels{kwargs['includechannels']}.fits"
        )

    elif kind == "final_ilc":
        return f"CN_needletILCmap_component_{kwargs['comp']}.fits"

    else:
        raise ValueError(f"Unknown filename kind: {kind}")

In [ ]:
def compare_files(fn, min_val=-4e-4, max_val=4e-4, suptitle=""):
    path_orig = Path(OLD_DIR) / fn
    path_new  = Path(NEW_DIR) / fn

    map_orig = HealpyMap().read(path_orig)[0]
    map_new  = HealpyMap().read(path_new)[0]

    print(map_orig.min(), map_new.min(), map_orig.max(), map_new.max())

    show_obs_logNxN([map_orig, map_new, map_new - map_orig], 
                    labels=["Old", "New", "Delta"],
                    min_val=min_val, max_val=max_val,
                    n_cols=3, n_rows=1, 
                    xsize=1200, dpi=100, 
                    suptitle=suptitle
                    )

In [ ]:
# Valid combinations are s=0, f_orig={0,...,7}
#                        s=1, f_orig={3,...,7}
#                        s=2, f_orig={4,...,7}
#                        s=3, f_orig={4,...,7}
compare_files(make_fn("needletcoeff", f_orig=0, s=0))

# Covariance maps

## Intermediate: smoothed maps

In [ ]:
def compare_files2(s, f, min_val=-4e-4, max_val=4e-4, suptitle=""):
    fn = f"smoothed_map_scale{s}_freq{f}.fits"
    path_orig = Path("/home/jim/VSCode_Projects_SPACE/pyilc") / fn
    path_new  = Path(NEW_DIR) / fn

    map_orig = HealpyMap().read(path_orig)[0]
    map_new  = HealpyMap().read(path_new)[0]

    print(map_orig.min(), map_new.min(), map_orig.max(), map_new.max())

    show_obs_logNxN([map_orig, map_new, map_new - map_orig], 
                    labels=["Old", "New", "Delta"],
                    min_val=min_val, max_val=max_val,
                    n_cols=3, n_rows=1, 
                    xsize=1200, dpi=100, 
                    suptitle=suptitle
                    )

In [ ]:
def compare_files2(s, f, min_val=-4e-4, max_val=4e-4, suptitle=""):
    fn = f"smoothed_map_scale{s}_freq{f}.fits"
    path_orig = Path("/home/jim/VSCode_Projects_SPACE/pyilc") / fn
    path_new  = Path(NEW_DIR) / fn

    map_orig = HealpyMap().read(path_orig)[0]
    map_new  = HealpyMap().read(path_new)[0]

    print(map_orig.min(), map_new.min(), map_orig.max(), map_new.max())

    show_obs_logNxN([map_orig, map_new, map_new - map_orig], 
                    labels=["Old", "New", "Delta"],
                    min_val=min_val, max_val=max_val,
                    n_cols=3, n_rows=1, 
                    xsize=1000, dpi=100, 
                    suptitle=suptitle
                    )

In [ ]:
# compare_files2(0,0, min_val=-0, max_val=5e-5)
# compare_files2(0,1, min_val=-0, max_val=4e-5)
# compare_files2(0,2, min_val=-0, max_val=3e-5)
# compare_files2(0,3, min_val=-0, max_val=6e-5)
# compare_files2(0,4, min_val=-0, max_val=1e-4)
# compare_files2(0,5, min_val=-0, max_val=1e-3)
# compare_files2(0,6, min_val=-0, max_val=2e-2)
# compare_files2(0,7, min_val=-1e-5, max_val=3)
# compare_files2(1,3, min_val=-1e-5, max_val=1e-5)
# compare_files2(1,4, min_val=-1e-5, max_val=1e-5)
# compare_files2(1,5, min_val=-1e-4, max_val=1e-4)
# compare_files2(1,6, min_val=-1e-2, max_val=1e-2)
# compare_files2(1,7, min_val=-1, max_val=1)
# compare_files2(2,4, min_val=-1e-7, max_val=1e-7)
# compare_files2(2,5, min_val=-1e-6, max_val=1e-6)
# compare_files2(2,6, min_val=-1e-5, max_val=1e-5)
# compare_files2(2,7, min_val=-1e-3, max_val=1e-3)
# compare_files2(3,4, min_val=-1e-9, max_val=1e-9)
# compare_files2(3,5, min_val=-1e-8, max_val=1e-8)
# compare_files2(3,6, min_val=-1e-7, max_val=1e-7)
# compare_files2(3,7, min_val=-1e-5, max_val=1e-5)

### Comparing masks

In [ ]:
import numpy as np

In [ ]:
def compare_masks(s, which_mask="degraded", min_val=0, max_val=1):
    if which_mask == "raw":
        fn = f"raw_mask_scale{s}.fits"
    elif which_mask == "degraded":
        fn = f"degraded_mask_scale{s}.fits"
    elif which_mask == "smoothed":
        fn = f"smoothed_mask_scale{s}.fits"
    elif which_mask == "fskyinv":
        fn = f"fsky_inv_scale{s}.fits"
    else:
        raise ValueError("mask type not known")
    suptitle = fn
    path_orig = Path("/home/jim/VSCode_Projects_SPACE/pyilc") / fn
    path_new  = Path(NEW_DIR) / fn

    map_orig = HealpyMap().read(path_orig)[0]
    map_new  = HealpyMap().read(path_new)[0]

    delta = map_orig-map_new
    print(map_orig.min(), map_new.min(), map_orig.max(), map_new.max())
    print(delta.min(),delta.max())
    print(np.where(delta!=0)[0].size)

    hp.mollview(delta, title=f"Delta, {fn}")

    if np.where(delta!=0)[0].size > 0:
        show_obs_logNxN([
                        map_orig, 
                        map_new, 
                        map_new - map_orig
                        ], 
                        labels=[
                            "Old", 
                            "New", 
                            "Delta"
                            ],
                        min_val=min_val, max_val=max_val,
                        n_cols=3, n_rows=1, 
                        xsize=1000, dpi=100, 
                        suptitle=suptitle
                        )

In [ ]:
# compare_masks(0, "raw")
# compare_masks(1, "raw")
# compare_masks(2, "raw")
# compare_masks(3, "raw")

In [ ]:
# compare_masks(0, "degraded")
# # compare_masks(1, "degraded")
# # compare_masks(2, "degraded")
# # compare_masks(3, "degraded")

In [ ]:
# compare_masks(0, "smoothed")
# # compare_masks(1, "smoothed")
# # compare_masks(2, "smoothed")
# # compare_masks(3, "smoothed")

In [ ]:
# compare_masks(0, "fskyinv")
# # compare_masks(1, "fskyinv")
# # compare_masks(2, "fskyinv")
# # compare_masks(3, "fskyinv")

## Comparing covariance maps

In [ ]:
# compare_files(make_fn("covmap", i_orig=4, j_orig=4, s=0), min_val=0, max_val=1.1e-8, suptitle="Freq 4x4, scale 0")
# compare_files(make_fn("covmap", i_orig=4, j_orig=5, s=1), min_val=-3e-9, max_val=1e-8, suptitle="Freq 4x5, scale 1")
# compare_files(make_fn("covmap", i_orig=4, j_orig=4, s=1), min_val=-3e-9, max_val=1e-8, suptitle="Freq 4x4, scale 1")
# compare_files(make_fn("covmap", i_orig=4, j_orig=4, s=2), min_val=0, max_val=1e-9, suptitle="Freq 4x4, scale 2")
compare_files(make_fn("covmap", i_orig=6, j_orig=7, s=2), min_val=0, max_val=1e-4, suptitle="Freq 6x7, scale 2")
# compare_files(make_fn("covmap", i_orig=4, j_orig=6, s=3), min_val=-1e-11, max_val=1e-10, suptitle="Freq 4x6, scale 3")
# compare_files(make_fn("covmap", i_orig=4, j_orig=7, s=3), min_val=0, max_val=1e-8, suptitle="Freq 4x7, scale 3")
# compare_files(make_fn("covmap", i_orig=5, j_orig=7, s=3), min_val=0, max_val=1e-7, suptitle="Freq 5x7, scale 3")
# compare_files(make_fn("covmap", i_orig=6, j_orig=7, s=3), min_val=0, max_val=1e-6, suptitle="Freq 6x7, scale 3")

In [ ]:
# compare_files(make_fn("per_scale_ilc", s=0, comp='CMB', includechannels='01234567'))
# compare_files(make_fn("per_scale_ilc", s=1, comp='CMB', includechannels='34567'), min_val=-3e-9, max_val=1e-8)
# compare_files(make_fn("per_scale_ilc", s=2, comp='CMB', includechannels='4567'), min_val=-3e-9, max_val=1e-8)
# compare_files(make_fn("per_scale_ilc", s=3, comp='CMB', includechannels='4567'), min_val=-3e-9, max_val=1e-8)

In [ ]:
import healpy as hp
import os

old = hp.read_map(os.path.join(OLD_DIR, "CN__needletcoeff_covmap_freq3_freq3_scale1.fits"))
new = hp.read_map(os.path.join(NEW_DIR, "CN__needletcoeff_covmap_freq3_freq3_scale1.fits"))
print("NSIDE old:", hp.get_nside(old), "  NSIDE new:", hp.get_nside(new))


In [ ]:
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Choose a single (freq, scale) pair that exists in both runs
f_orig, s = 3, 1   # example: freq3, scale1

fname = f"CN__needletcoeffmap_freq{f_orig}_scale{s}.fits"
old_path = Path(OLD_DIR) / fname
new_path = Path(NEW_DIR) / fname

m_old = hp.read_map(old_path, field=0)
m_new = hp.read_map(new_path, field=0)

# remove monopole & dipole just to focus on beam shapes
m_old = hp.remove_dipole(m_old, fitval=True)[0]
m_new = hp.remove_dipole(m_new, fitval=True)[0]

lmax = min(3 * hp.get_nside(m_old) - 1, 3000)
cl_old = hp.anafast(m_old, lmax=lmax)
cl_new = hp.anafast(m_new, lmax=lmax)

ells = np.arange(len(cl_old))

plt.figure(figsize=(8,5))
plt.loglog(ells, cl_old, label="Old (PyILC)")
plt.loglog(ells, cl_new, label="New (Simpler ILc)", linestyle="--")
plt.loglog(ells, cl_new / cl_old, label="Ratio (new/old)", linestyle="--", color="gray")
plt.xlabel(r"$\ell$")
plt.ylabel(r"$C_\ell$")
plt.title(f"Needlet coeff map power spectra: freq={f_orig}, scale={s}")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Optional: estimate effective beam mismatch
ratio = np.nanmean((cl_new / cl_old)[50:500])  # average in mid-ℓ range
print(f"Mean C_ell(new)/C_ell(old) between 50<ℓ<500: {ratio:.3f}")


In [ ]:
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

f_i, f_j, s = 6, 6, 3
fname = f"CN__needletcoeff_covmap_freq{f_i}_freq{f_j}_scale{s}.fits"
old = hp.read_map(Path(OLD_DIR)/fname, field=0)
new = hp.read_map(Path(NEW_DIR)/fname, field=0)

# remove mean
old -= np.nanmean(old[old != hp.UNSEEN])
new -= np.nanmean(new[new != hp.UNSEEN])

lmax = min(3*hp.get_nside(old)-1, 3000)
cl_old = hp.anafast(old, lmax=lmax)
cl_new = hp.anafast(new, lmax=lmax)
ells = np.arange(len(cl_old))

plt.figure(figsize=(8,5))
plt.loglog(ells, cl_old, label="Old (PyILC)")
plt.loglog(ells, cl_new, label="New (Simpler ILC)")
plt.xlabel(r"$\ell$")
plt.ylabel(r"$C_\ell$")
plt.title(f"Covmap power spectra: freq{f_i}, freq{f_j}, scale={s}")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
total = 0
for i in range(12):
    nside = 2**i
    npix = hp.nside2npix(nside)
    print(nside, npix)
    total += npix

print(total)
print(total / npix)